## Model Comparison for Second-Phase Structural Prediction

This notebook compares seven regressors for `Grain_2`, `a_2`, `b_2`,
and `c_2` using iteration 6 datasets: XGBoost, Ridge, Gaussian Process,
Kernel Ridge, Random Forest, Gradient Boosting, and TabPFN.

### Workflow

1. Aggregate records by compound, retaining the first feature values
   and averaging target values.
2. Split each target dataset into training and test sets (80/20).
3. Select features using XGBoost: 12 for Grain_2 and a_2,
   13 for b_2, and 14 for c_2.
4. Evaluate model configurations using five-fold cross-validation
   with MAE as the selection metric.
5. Export comparison metrics ranked by test R².

Grain_2 training, evaluation, and exported predictions use the log1p
scale. Lattice parameters use their original scales.
Feature selection occurs before cross-validation.
TabPFN runs on CPU using the specified local checkpoint.

Each target section can run independently. Complete its cells in order,
including the comparison export, before switching targets.

## grain2

In [1]:
# ============================================================
# 1. Import packages
# ============================================================

import os
import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from tabpfn import TabPFNRegressor

from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor
)
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    make_scorer,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.svm import SVR

from sklearn.gaussian_process import (
    GaussianProcessRegressor
)
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    RBF,
    WhiteKernel
)

D:\app\miniconda\envs\mp311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================
# 2. Basic settings
# ============================================================

DATA_FILE = "data/compound_6_Structural_Grain2.csv"

TARGET_COL = "Grain_2"
GROUP_COL = "compound"

N_SELECTED_FEATURES = 12
TEST_SIZE = 0.20
RANDOM_STATE = 8

TABPFN_MODEL_PATH = (
    "tabpfn_models/tabpfn-v2-regressor.ckpt"
)

OUTPUT_DIR = "model_comparison/Grain2"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

In [3]:
# ============================================================
# 3. Data preparation
# ============================================================

data = pd.read_csv(DATA_FILE)

data = data.drop(
    columns=[
        col for col in data.columns
        if str(col).startswith("Unnamed:")
    ],
    errors="ignore"
)

feature_cols = [
    col for col in data.columns
    if col not in [GROUP_COL, TARGET_COL]
]

aggregation_rules = {
    col: "first"
    for col in feature_cols
}

aggregation_rules[TARGET_COL] = "mean"

data = (
    data
    .groupby(GROUP_COL, as_index=False)
    .agg(aggregation_rules)
)

train_data, test_data = train_test_split(
    data,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

train_data = train_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

X_train_all = train_data[feature_cols]
X_test_all = test_data[feature_cols]

y_train_original = (
    train_data[TARGET_COL]
    .to_numpy()
)

y_test_original = (
    test_data[TARGET_COL]
    .to_numpy()
)

y_train_log = np.log1p(
    y_train_original
)

y_test_log = np.log1p(
    y_test_original
)

print("Total samples:", len(data))
print("Training samples:", len(train_data))
print("Test samples:", len(test_data))
print("Candidate features:", len(feature_cols))

Total samples: 55
Training samples: 44
Test samples: 11
Candidate features: 25


In [4]:
# ============================================================
# 4. XGBoost feature selection
# ============================================================

feature_selector = XGBRegressor(
    n_estimators=800,
#     max_depth=2,
#     learning_rate=0.6,
    importance_type="gain",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=0
)

feature_selector.fit(
    X_train_all,
    y_train_log
)

feature_importance = pd.Series(
    feature_selector.feature_importances_,
    index=feature_cols
).sort_values(
    ascending=False
)

selected_features = (
    feature_importance
    .head(N_SELECTED_FEATURES)
    .index
    .tolist()
)

X_train = train_data[selected_features]
X_test = test_data[selected_features]

print("\nSelected features:")

for index, feature in enumerate(
    selected_features,
    start=1
):
    print(f"{index}. {feature}")


Selected features:
1. C_AW
2. AB_AR
3. AB_EN
4. AB_CR
5. AB_Nd
6. AB_EC
7. AB_FIE
8. AB_Nv
9. AB_AW
10. AB_TC
11. AB_AN
12. AB_EA


In [5]:
# ============================================================
# 5. Cross-validation
# ============================================================

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


# ============================================================
# 6. Model training and evaluation function
# ============================================================

def train_and_evaluate(
    model_name,
    model,
    param_grid,
    n_jobs=-1
):

    search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring="neg_mean_absolute_error",
        cv=cv,
        n_jobs=n_jobs,
        refit=True
    )

    search.fit(
        X_train,
        y_train_log
    )

    best_model = search.best_estimator_

    train_prediction_log = best_model.predict(
        X_train
    )

    test_prediction_log = best_model.predict(
        X_test
    )

    train_r2_log = r2_score(
        y_train_log,
        train_prediction_log
    )

    train_mae_log = mean_absolute_error(
        y_train_log,
        train_prediction_log
    )

    train_rmse_log = np.sqrt(
        mean_squared_error(
            y_train_log,
            train_prediction_log
        )
    )

    test_r2_log = r2_score(
        y_test_log,
        test_prediction_log
    )

    test_mae_log = mean_absolute_error(
        y_test_log,
        test_prediction_log
    )

    test_rmse_log = np.sqrt(
        mean_squared_error(
            y_test_log,
            test_prediction_log
        )
    )

    print(f"\n===== {model_name} =====")

    print(
        "Best parameters:",
        search.best_params_
    )

    print(
        f"Five-fold CV MAE: "
        f"{-search.best_score_:.6f}"
    )

    print(
        f"Train: R2={train_r2_log:.6f} | "
        f"MAE={train_mae_log:.6f} | "
        f"RMSE={train_rmse_log:.6f}"
    )

    print(
        f"Test: R2={test_r2_log:.6f} | "
        f"MAE={test_mae_log:.6f} | "
        f"RMSE={test_rmse_log:.6f}"
    )

    result = {
        "Model": model_name,
        "CV_MAE_log": -search.best_score_,
        "Train_R2_log": train_r2_log,
        "Train_MAE_log": train_mae_log,
        "Train_RMSE_log": train_rmse_log,
        "Test_R2_log": test_r2_log,
        "Test_MAE_log": test_mae_log,
        "Test_RMSE_log": test_rmse_log,
        "Best_parameters": str(
            search.best_params_
        )
    }

    prediction_result = pd.DataFrame({
        GROUP_COL: test_data[GROUP_COL],
        "y_true_log": y_test_log,
        "y_pred_log": test_prediction_log
    })

    prediction_result.to_csv(
        f"{OUTPUT_DIR}/{model_name}_test_predictions.csv",
        index=False
    )

    return best_model, result

In [6]:
# ============================================================
# 7. XGBoost
# ============================================================

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=8,
    n_jobs=1,
    verbosity=0
)

xgb_grid = {
    "n_estimators": [
        30,
        50,
        100,
        200,
        300
    ],
    "max_depth": [
        3,
        6,
        8,
        9
    ],
    "learning_rate": [
        0.01,
        0.03,
        0.05,
    ]
}

best_xgb, xgb_result = train_and_evaluate(
    model_name="XGBoost",
    model=xgb_model,
    param_grid=xgb_grid
)



===== XGBoost =====
Best parameters: {'learning_rate': 0.03, 'max_depth': 8, 'n_estimators': 200}
Five-fold CV MAE: 0.534974
Train: R2=0.999028 | MAE=0.016968 | RMSE=0.024687
Test: R2=0.788676 | MAE=0.430517 | RMSE=0.505916


In [7]:
# ============================================================
# 8. Ridge
# ============================================================

ridge_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        Ridge()
    )
])

ridge_grid = {
    "model__alpha": [
        0.001,
        0.01,
        0.1,
        1,
        10,
        100
    ]
}

best_ridge, ridge_result = train_and_evaluate(
    model_name="Ridge",
    model=ridge_model,
    param_grid=ridge_grid
)



===== Ridge =====
Best parameters: {'model__alpha': 1}
Five-fold CV MAE: 0.578201
Train: R2=0.282098 | MAE=0.522407 | RMSE=0.670978
Test: R2=0.623652 | MAE=0.456930 | RMSE=0.675147


In [8]:
# ============================================================
# 9. Gaussian process regression
# ============================================================

gpr_kernel = (
    ConstantKernel(
        constant_value=1.0,
        constant_value_bounds=(1e-2, 1e2)
    )
    * RBF(
        length_scale=1.0,
        length_scale_bounds=(1e-2, 1e2)
    )
    + WhiteKernel(
        noise_level=1e-3,
        noise_level_bounds=(1e-6, 1e0)
    )
)

gpr_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        GaussianProcessRegressor(
            kernel=gpr_kernel,
            normalize_y=True,
            n_restarts_optimizer=5,
            random_state=RANDOM_STATE
        )
    )
])

gpr_grid = {
    "model__alpha": [
        1e-8,
        1e-6,
        1e-4,
        1e-2
    ]
}

best_gpr, gpr_result = train_and_evaluate(
    model_name="GPR",
    model=gpr_model,
    param_grid=gpr_grid
)


===== GPR =====
Best parameters: {'model__alpha': 0.01}
Five-fold CV MAE: 0.588795
Train: R2=0.965436 | MAE=0.118445 | RMSE=0.147227
Test: R2=0.205256 | MAE=0.631639 | RMSE=0.981109


In [9]:
# ============================================================
# 10. Kernel ridge
# ============================================================

krr_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        KernelRidge(
            kernel="rbf"
        )
    )
])

krr_grid = {
    "model__alpha": [
        0.001,
        0.01,
        0.1,
        1,
        10
    ],
    "model__gamma": [
        0.001,
        0.01,
        0.1,
        1
    ]
}

best_krr, krr_result = train_and_evaluate(
    model_name="Kernel Ridge",
    model=krr_model,
    param_grid=krr_grid
)



===== Kernel Ridge =====
Best parameters: {'model__alpha': 0.001, 'model__gamma': 0.001}
Five-fold CV MAE: 0.582292
Train: R2=0.330053 | MAE=0.498654 | RMSE=0.648181
Test: R2=0.683838 | MAE=0.423893 | RMSE=0.618811


In [10]:
# ============================================================
# 11. Random forest
# ============================================================

rf_model = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=1
)

rf_grid = {
    "n_estimators": [
        100,
        300,
        500
    ],
    "max_depth": [
        None,
        3,
        5
    ],
    "min_samples_leaf": [
        1,
        2,
        4
    ]
}

best_rf, rf_result = train_and_evaluate(
    model_name="Random Forest",
    model=rf_model,
    param_grid=rf_grid
)



===== Random Forest =====
Best parameters: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 500}
Five-fold CV MAE: 0.575750
Train: R2=0.863833 | MAE=0.216348 | RMSE=0.292221
Test: R2=0.651052 | MAE=0.465309 | RMSE=0.650105


In [11]:
# ============================================================
# 12. Gradient boosting
# ============================================================

gbdt_model = GradientBoostingRegressor(
    random_state=RANDOM_STATE
)

gbdt_grid = {
    "n_estimators": [
        50,
        100,
        200
    ],
    "learning_rate": [
        0.03,
        0.05,
        0.10
    ],
    "max_depth": [
        1,
        2,
        3
    ],
    "min_samples_leaf": [
        1,
        2,
        4
    ]
}

best_gbdt, gbdt_result = train_and_evaluate(
    model_name="Gradient Boosting",
    model=gbdt_model,
    param_grid=gbdt_grid
)



===== Gradient Boosting =====
Best parameters: {'learning_rate': 0.05, 'max_depth': 2, 'min_samples_leaf': 4, 'n_estimators': 200}
Five-fold CV MAE: 0.584717
Train: R2=0.938523 | MAE=0.147165 | RMSE=0.196350
Test: R2=0.387909 | MAE=0.630638 | RMSE=0.861017


In [12]:
# ============================================================
# 13. TabPFN
# ============================================================

tabpfn_model = TabPFNRegressor(
    device="cpu",
    model_path=TABPFN_MODEL_PATH,
    random_state=RANDOM_STATE
)

tabpfn_grid = {}

best_tabpfn, tabpfn_result = train_and_evaluate(
    model_name="TabPFN",
    model=tabpfn_model,
    param_grid=tabpfn_grid,
    n_jobs=1
)


===== TabPFN =====
Best parameters: {}
Five-fold CV MAE: 0.591158
Train: R2=0.548196 | MAE=0.439535 | RMSE=0.532293
Test: R2=0.667585 | MAE=0.439648 | RMSE=0.634518


In [13]:
# ============================================================
# 14. Model comparison
# ============================================================

comparison_results = pd.DataFrame([
    xgb_result,
    ridge_result,
    gpr_result,
    rf_result,
    gbdt_result,
    tabpfn_result,
    krr_result
])

comparison_results = (
    comparison_results
    .sort_values(
        "Test_R2_log",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nFinal model comparison:")

print(
    comparison_results[
        [
            "Model",
            "CV_MAE_log",
            "Test_R2_log",
            "Test_MAE_log",
            "Test_RMSE_log"
        ]
    ]
)

comparison_results.to_csv(
    f"{OUTPUT_DIR}/model_comparison_results_log.csv",
    index=False
)


Final model comparison:
               Model  CV_MAE_log  Test_R2_log  Test_MAE_log  Test_RMSE_log
0            XGBoost    0.534974     0.788676      0.430517       0.505916
1       Kernel Ridge    0.582292     0.683838      0.423893       0.618811
2             TabPFN    0.591158     0.667585      0.439648       0.634518
3      Random Forest    0.575750     0.651052      0.465309       0.650105
4              Ridge    0.578201     0.623652      0.456930       0.675147
5  Gradient Boosting    0.584717     0.387909      0.630638       0.861017
6                GPR    0.588795     0.205256      0.631639       0.981109


## a2

In [1]:
# ============================================================
# 1. Import packages
# ============================================================

import os
import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from tabpfn import TabPFNRegressor

from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor
)
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.svm import SVR

from sklearn.gaussian_process import (
    GaussianProcessRegressor
)
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    RBF,
    WhiteKernel
)

D:\app\miniconda\envs\mp311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================
# 2. Basic settings
# ============================================================

DATA_FILE = "data/compound_6_Structural_a2.csv"

TARGET_COL = "a_2"
GROUP_COL = "compound"

N_SELECTED_FEATURES = 12
TEST_SIZE = 0.20
RANDOM_STATE = 1100

TABPFN_MODEL_PATH = (
    "tabpfn_models/tabpfn-v2-regressor.ckpt"
)

OUTPUT_DIR = "model_comparison/a2"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


In [3]:
# ============================================================
# 3. Data preparation
# ============================================================

data = pd.read_csv(DATA_FILE)

data = data.drop(
    columns=[
        col for col in data.columns
        if str(col).startswith("Unnamed:")
    ],
    errors="ignore"
)

feature_cols = [
    col for col in data.columns
    if col not in [GROUP_COL, TARGET_COL]
]

aggregation_rules = {
    col: "first"
    for col in feature_cols
}

aggregation_rules[TARGET_COL] = "mean"

data = (
    data
    .groupby(GROUP_COL, as_index=False)
    .agg(aggregation_rules)
)

train_data, test_data = train_test_split(
    data,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

train_data = train_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

X_train_all = train_data[feature_cols]
X_test_all = test_data[feature_cols]

y_train = (
    train_data[TARGET_COL]
    .to_numpy()
)

y_test = (
    test_data[TARGET_COL]
    .to_numpy()
)

print("Total samples:", len(data))
print("Training samples:", len(train_data))
print("Test samples:", len(test_data))
print("Candidate features:", len(feature_cols))

Total samples: 55
Training samples: 44
Test samples: 11
Candidate features: 25


In [4]:
# ============================================================
# 4. XGBoost feature selection
# ============================================================

feature_selector = XGBRegressor(
    n_estimators=100,
    max_depth=2,
    learning_rate=0.05,
    importance_type="gain",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=0
)

feature_selector.fit(
    X_train_all,
    y_train
)

feature_importance = pd.Series(
    feature_selector.feature_importances_,
    index=feature_cols
).sort_values(
    ascending=False
)

selected_features = (
    feature_importance
    .head(N_SELECTED_FEATURES)
    .index
    .tolist()
)

X_train = train_data[selected_features]
X_test = test_data[selected_features]

print("\nSelected features:")

for index, feature in enumerate(
    selected_features,
    start=1
):
    print(f"{index}. {feature}")


Selected features:
1. C_EA
2. AB_EN
3. C_SIE
4. AB_CR
5. AB_FIE
6. AB_TC
7. AB_Nd
8. AB_SIE
9. AB_D
10. AB_Nv
11. AB_EA
12. AB_EC


In [5]:
# ============================================================
# 5. Cross-validation
# ============================================================

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


# ============================================================
# 6. Model training and evaluation function
# ============================================================

def train_and_evaluate(
    model_name,
    model,
    param_grid,
    n_jobs=-1
):

    search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring="neg_mean_absolute_error",
        cv=cv,
        n_jobs=n_jobs,
        refit=True
    )

    search.fit(
        X_train,
        y_train
    )

    best_model = search.best_estimator_

    train_prediction = best_model.predict(
        X_train
    )

    test_prediction = best_model.predict(
        X_test
    )

    train_r2 = r2_score(
        y_train,
        train_prediction
    )

    train_mae = mean_absolute_error(
        y_train,
        train_prediction
    )

    train_rmse = np.sqrt(
        mean_squared_error(
            y_train,
            train_prediction
        )
    )

    test_r2 = r2_score(
        y_test,
        test_prediction
    )

    test_mae = mean_absolute_error(
        y_test,
        test_prediction
    )

    test_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            test_prediction
        )
    )

    print(f"\n===== {model_name} =====")
    print("Best parameters:", search.best_params_)
    print(
        f"Five-fold CV MAE: "
        f"{-search.best_score_:.6f}"
    )
    print(
        f"Train: R2={train_r2:.6f} | "
        f"MAE={train_mae:.6f} | "
        f"RMSE={train_rmse:.6f}"
    )
    print(
        f"Test: R2={test_r2:.6f} | "
        f"MAE={test_mae:.6f} | "
        f"RMSE={test_rmse:.6f}"
    )

    result = {
        "Model": model_name,
        "CV_MAE": -search.best_score_,
        "Train_R2": train_r2,
        "Train_MAE": train_mae,
        "Train_RMSE": train_rmse,
        "Test_R2": test_r2,
        "Test_MAE": test_mae,
        "Test_RMSE": test_rmse,
        "Best_parameters": str(
            search.best_params_
        )
    }

    return best_model, result

In [6]:
# ============================================================
# 7. XGBoost
# ============================================================

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=0
)

xgb_grid = {
    "n_estimators": [
        50,
        100,
        200,
        300,
    ],
    "max_depth": [
        3,
        6,
        8,
        9,
        10
    ],
    "learning_rate": [
        0.05,
        0.10,
        0.20,
    ]
}

best_xgb, xgb_result = train_and_evaluate(
    model_name="XGBoost",
    model=xgb_model,
    param_grid=xgb_grid
)


===== XGBoost =====
Best parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
Five-fold CV MAE: 0.974273
Train: R2=0.998120 | MAE=0.065527 | RMSE=0.096418
Test: R2=0.823684 | MAE=0.613160 | RMSE=0.777129


In [7]:
# ============================================================
# 8. Ridge
# ============================================================

ridge_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        Ridge()
    )
])

ridge_grid = {
    "model__alpha": [
        0.001,
        0.01,
        0.1,
        1,
        10,
        100
    ]
}

best_ridge, ridge_result = train_and_evaluate(
    model_name="Ridge",
    model=ridge_model,
    param_grid=ridge_grid
)


===== Ridge =====
Best parameters: {'model__alpha': 0.001}
Five-fold CV MAE: 1.488212
Train: R2=0.641496 | MAE=0.992777 | RMSE=1.331602
Test: R2=0.306456 | MAE=1.208233 | RMSE=1.541288


In [8]:
# ============================================================
# 9. Gaussian process regression
# ============================================================

gpr_kernel = (
    ConstantKernel(
        constant_value=1.0,
        constant_value_bounds=(1e-2, 1e2)
    )
    * RBF(
        length_scale=1.0,
        length_scale_bounds=(1e-2, 1e2)
    )
    + WhiteKernel(
        noise_level=1e-3,
        noise_level_bounds=(1e-6, 1e0)
    )
)

gpr_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        GaussianProcessRegressor(
            kernel=gpr_kernel,
            normalize_y=True,
            n_restarts_optimizer=5,
            random_state=RANDOM_STATE
        )
    )
])

gpr_grid = {
    "model__alpha": [
        1e-8,
        1e-6,
        1e-4,
        1e-2
    ]
}

best_gpr, gpr_result = train_and_evaluate(
    model_name="GPR",
    model=gpr_model,
    param_grid=gpr_grid
)


===== GPR =====
Best parameters: {'model__alpha': 1e-08}
Five-fold CV MAE: 1.217137
Train: R2=0.873214 | MAE=0.545606 | RMSE=0.791889
Test: R2=0.563790 | MAE=1.074315 | RMSE=1.222347


In [9]:
# ============================================================
# 10. Kernel ridge
# ============================================================

krr_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        KernelRidge(
            kernel="rbf"
        )
    )
])

krr_grid = {
    "model__alpha": [
        0.001,
        0.01,
        0.1,
        1,
        10
    ],
    "model__gamma": [
        0.001,
        0.01,
        0.1,
        1
    ]
}

best_krr, krr_result = train_and_evaluate(
    model_name="Kernel Ridge",
    model=krr_model,
    param_grid=krr_grid
)


===== Kernel Ridge =====
Best parameters: {'model__alpha': 0.01, 'model__gamma': 0.1}
Five-fold CV MAE: 1.103920
Train: R2=0.857656 | MAE=0.546471 | RMSE=0.839067
Test: R2=0.598042 | MAE=1.003027 | RMSE=1.173376


In [10]:
# ============================================================
# 11. Random forest
# ============================================================

rf_model = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=1
)

rf_grid = {
    "n_estimators": [
        100,
        300,
        500
    ],
    "max_depth": [
        None,
        3,
        5
    ],
    "min_samples_leaf": [
        1,
        2,
        4
    ]
}

best_rf, rf_result = train_and_evaluate(
    model_name="Random Forest",
    model=rf_model,
    param_grid=rf_grid
)



===== Random Forest =====
Best parameters: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 300}
Five-fold CV MAE: 1.158020
Train: R2=0.946079 | MAE=0.367274 | RMSE=0.516425
Test: R2=0.760927 | MAE=0.800856 | RMSE=0.904924


In [11]:
# ============================================================
# 12. Gradient boosting
# ============================================================

gbdt_model = GradientBoostingRegressor(
    random_state=RANDOM_STATE
)

gbdt_grid = {
    "n_estimators": [
        50,
        100,
        200
    ],
    "learning_rate": [
        0.03,
        0.05,
        0.10
    ],
    "max_depth": [
        1,
        2,
        3
    ]
}

best_gbdt, gbdt_result = train_and_evaluate(
    model_name="Gradient Boosting",
    model=gbdt_model,
    param_grid=gbdt_grid
)



===== Gradient Boosting =====
Best parameters: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200}
Five-fold CV MAE: 0.988631
Train: R2=0.999692 | MAE=0.030008 | RMSE=0.039029
Test: R2=0.804029 | MAE=0.594468 | RMSE=0.819300


In [12]:
# ============================================================
# 13. TabPFN
# ============================================================

tabpfn_model = TabPFNRegressor(
    device="cpu",
    model_path=TABPFN_MODEL_PATH,
    random_state=RANDOM_STATE
)

tabpfn_grid = {}

best_tabpfn, tabpfn_result = train_and_evaluate(
    model_name="TabPFN",
    model=tabpfn_model,
    param_grid=tabpfn_grid,
    n_jobs=1
)



===== TabPFN =====
Best parameters: {}
Five-fold CV MAE: 1.250389
Train: R2=0.922203 | MAE=0.440377 | RMSE=0.620309
Test: R2=0.752076 | MAE=0.833260 | RMSE=0.921524


In [14]:
# ============================================================
# 14. Model comparison
# ============================================================

comparison_results = pd.DataFrame([
    xgb_result,
    ridge_result,
    gpr_result,
    rf_result,
    gbdt_result,
    tabpfn_result,
    krr_result,

])

comparison_results = (
    comparison_results
    .sort_values(
        "Test_R2",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nFinal model comparison:")

print(
    comparison_results[
        [
            "Model",
            "CV_MAE",
            "Test_R2",
            "Test_MAE",
            "Test_RMSE"
        ]
    ]
)

comparison_results.to_csv(
    f"{OUTPUT_DIR}/model_comparison_results.csv",
    index=False
)


Final model comparison:
               Model    CV_MAE   Test_R2  Test_MAE  Test_RMSE
0            XGBoost  0.974273  0.823684  0.613160   0.777129
1  Gradient Boosting  0.988631  0.804029  0.594468   0.819300
2      Random Forest  1.158020  0.760927  0.800856   0.904924
3             TabPFN  1.250389  0.752076  0.833260   0.921524
4       Kernel Ridge  1.103920  0.598042  1.003027   1.173376
5                GPR  1.217137  0.563790  1.074315   1.222347
6              Ridge  1.488212  0.306456  1.208233   1.541288


## b2

In [1]:
# ============================================================
# 1. Import packages
# ============================================================

import os
import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from tabpfn import TabPFNRegressor

from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor
)
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.svm import SVR

from sklearn.gaussian_process import (
    GaussianProcessRegressor
)
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    RBF,
    WhiteKernel
)


D:\app\miniconda\envs\mp311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================
# 2. Basic settings
# ============================================================

DATA_FILE = "data/compound_6_Structural_b2.csv"

TARGET_COL = "b_2"
GROUP_COL = "compound"

N_SELECTED_FEATURES = 13
TEST_SIZE = 0.20
RANDOM_STATE = 110

TABPFN_MODEL_PATH = (
    "tabpfn_models/tabpfn-v2-regressor.ckpt"
)

OUTPUT_DIR = "model_comparison/b2"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

In [3]:
# ============================================================
# 3. Data preparation
# ============================================================

data = pd.read_csv(DATA_FILE)

data = data.drop(
    columns=[
        col for col in data.columns
        if str(col).startswith("Unnamed:")
    ],
    errors="ignore"
)

feature_cols = [
    col for col in data.columns
    if col not in [GROUP_COL, TARGET_COL]
]

aggregation_rules = {
    col: "first"
    for col in feature_cols
}

aggregation_rules[TARGET_COL] = "mean"

data = (
    data
    .groupby(GROUP_COL, as_index=False)
    .agg(aggregation_rules)
)

train_data, test_data = train_test_split(
    data,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

train_data = train_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

X_train_all = train_data[feature_cols]
X_test_all = test_data[feature_cols]

y_train = (
    train_data[TARGET_COL]
    .to_numpy()
)

y_test = (
    test_data[TARGET_COL]
    .to_numpy()
)

print("Total samples:", len(data))
print("Training samples:", len(train_data))
print("Test samples:", len(test_data))
print("Candidate features:", len(feature_cols))

Total samples: 55
Training samples: 44
Test samples: 11
Candidate features: 25


In [4]:
# ============================================================
# 4. XGBoost feature selection
# ============================================================

feature_selector = XGBRegressor(
    n_estimators=100,
    max_depth=2,
    learning_rate=0.05,
    importance_type="gain",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=0
)

feature_selector.fit(
    X_train_all,
    y_train
)

feature_importance = pd.Series(
    feature_selector.feature_importances_,
    index=feature_cols
).sort_values(
    ascending=False
)

selected_features = (
    feature_importance
    .head(N_SELECTED_FEATURES)
    .index
    .tolist()
)

X_train = train_data[selected_features]
X_test = test_data[selected_features]

print("\nSelected features:")

for index, feature in enumerate(
    selected_features,
    start=1
):
    print(f"{index}. {feature}")



Selected features:
1. C_EA
2. AB_EN
3. AB_FIE
4. AB_Nd
5. AB_D
6. AB_CR
7. AB_EC
8. AB_Nv
9. AB_AW
10. AB_AR
11. C_CR
12. AB_SIE
13. AB_AN


In [5]:
# ============================================================
# 5. Cross-validation
# ============================================================

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


# ============================================================
# 6. Model training and evaluation function
# ============================================================

def train_and_evaluate(
    model_name,
    model,
    param_grid,
    n_jobs=-1
):

    search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring="neg_mean_absolute_error",
        cv=cv,
        n_jobs=n_jobs,
        refit=True
    )

    search.fit(
        X_train,
        y_train
    )

    best_model = search.best_estimator_

    train_prediction = best_model.predict(
        X_train
    )

    test_prediction = best_model.predict(
        X_test
    )

    train_r2 = r2_score(
        y_train,
        train_prediction
    )

    train_mae = mean_absolute_error(
        y_train,
        train_prediction
    )

    train_rmse = np.sqrt(
        mean_squared_error(
            y_train,
            train_prediction
        )
    )

    test_r2 = r2_score(
        y_test,
        test_prediction
    )

    test_mae = mean_absolute_error(
        y_test,
        test_prediction
    )

    test_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            test_prediction
        )
    )

    print(f"\n===== {model_name} =====")
    print("Best parameters:", search.best_params_)
    print(
        f"Five-fold CV MAE: "
        f"{-search.best_score_:.6f}"
    )
    print(
        f"Train: R2={train_r2:.6f} | "
        f"MAE={train_mae:.6f} | "
        f"RMSE={train_rmse:.6f}"
    )
    print(
        f"Test: R2={test_r2:.6f} | "
        f"MAE={test_mae:.6f} | "
        f"RMSE={test_rmse:.6f}"
    )

    result = {
        "Model": model_name,
        "CV_MAE": -search.best_score_,
        "Train_R2": train_r2,
        "Train_MAE": train_mae,
        "Train_RMSE": train_rmse,
        "Test_R2": test_r2,
        "Test_MAE": test_mae,
        "Test_RMSE": test_rmse,
        "Best_parameters": str(
            search.best_params_
        )
    }

    return best_model, result

In [6]:
# ============================================================
# 7. XGBoost
# ============================================================

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=0
)

xgb_grid = {
    "n_estimators": [
        50,
        100,
        200
    ],
    "max_depth": [
        1,
        2,
        3
    ],
    "learning_rate": [
        0.03,
        0.05,
        0.10
    ]
}

best_xgb, xgb_result = train_and_evaluate(
    model_name="XGBoost",
    model=xgb_model,
    param_grid=xgb_grid
)


===== XGBoost =====
Best parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}
Five-fold CV MAE: 0.742416
Train: R2=0.999960 | MAE=0.010117 | RMSE=0.013501
Test: R2=0.844390 | MAE=0.757389 | RMSE=0.981133


In [7]:
# ============================================================
# 8. Ridge
# ============================================================

ridge_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        Ridge()
    )
])

ridge_grid = {
    "model__alpha": [
        0.001,
        0.01,
        0.1,
        1,
        10,
        100
    ]
}

best_ridge, ridge_result = train_and_evaluate(
    model_name="Ridge",
    model=ridge_model,
    param_grid=ridge_grid
)



===== Ridge =====
Best parameters: {'model__alpha': 1}
Five-fold CV MAE: 1.215305
Train: R2=0.552422 | MAE=1.045892 | RMSE=1.421703
Test: R2=0.309663 | MAE=1.872506 | RMSE=2.066520


In [8]:
# ============================================================
# 9. Gaussian process regression
# ============================================================

gpr_kernel = (
    ConstantKernel(
        constant_value=1.0,
        constant_value_bounds=(1e-2, 1e2)
    )
    * RBF(
        length_scale=1.0,
        length_scale_bounds=(1e-2, 1e2)
    )
    + WhiteKernel(
        noise_level=1e-3,
        noise_level_bounds=(1e-6, 1e0)
    )
)

gpr_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        GaussianProcessRegressor(
            kernel=gpr_kernel,
            normalize_y=True,
            n_restarts_optimizer=5,
            random_state=RANDOM_STATE
        )
    )
])

gpr_grid = {
    "model__alpha": [
        1e-8,
        1e-6,
        1e-4,
        1e-2
    ]
}

best_gpr, gpr_result = train_and_evaluate(
    model_name="GPR",
    model=gpr_model,
    param_grid=gpr_grid
)


===== GPR =====
Best parameters: {'model__alpha': 0.01}
Five-fold CV MAE: 1.085827
Train: R2=0.978004 | MAE=0.241755 | RMSE=0.315174
Test: R2=0.758492 | MAE=1.015694 | RMSE=1.222292


In [9]:
# ============================================================
# 10. Kernel ridge
# ============================================================

krr_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        KernelRidge(
            kernel="rbf"
        )
    )
])

krr_grid = {
    "model__alpha": [
        0.01,
        0.1,
        1,
    ],
    "model__gamma": [
        0.001,
        0.01,
    ]
}

best_krr, krr_result = train_and_evaluate(
    model_name="Kernel Ridge",
    model=krr_model,
    param_grid=krr_grid
)



===== Kernel Ridge =====
Best parameters: {'model__alpha': 0.01, 'model__gamma': 0.01}
Five-fold CV MAE: 1.040894
Train: R2=0.730146 | MAE=0.785963 | RMSE=1.103923
Test: R2=0.612359 | MAE=1.355041 | RMSE=1.548545


In [10]:
# ============================================================
# 11. Random forest
# ============================================================

rf_model = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=1
)

rf_grid = {
    "n_estimators": [
        100,
        300,
        500
    ],
    "max_depth": [
        None,
        3,
        5
    ],
    "min_samples_leaf": [
        1,
        2,
        4
    ]
}

best_rf, rf_result = train_and_evaluate(
    model_name="Random Forest",
    model=rf_model,
    param_grid=rf_grid
)



===== Random Forest =====
Best parameters: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 500}
Five-fold CV MAE: 0.893546
Train: R2=0.953120 | MAE=0.304052 | RMSE=0.460117
Test: R2=0.782593 | MAE=1.008776 | RMSE=1.159701


In [11]:
# ============================================================
# 12. Gradient boosting
# ============================================================

gbdt_model = GradientBoostingRegressor(
    random_state=RANDOM_STATE
)

gbdt_grid = {
    "n_estimators": [
        50,
        100,
        200
    ],
    "learning_rate": [
        0.03,
        0.05,
        0.06
    ],
    "max_depth": [
        1,
        2,
        3
    ],
    "min_samples_leaf": [
        2,
        3,
        4
    ]
}

best_gbdt, gbdt_result = train_and_evaluate(
    model_name="Gradient Boosting",
    model=gbdt_model,
    param_grid=gbdt_grid
)



===== Gradient Boosting =====
Best parameters: {'learning_rate': 0.06, 'max_depth': 3, 'min_samples_leaf': 2, 'n_estimators': 200}
Five-fold CV MAE: 0.910095
Train: R2=0.999699 | MAE=0.028389 | RMSE=0.036871
Test: R2=0.827132 | MAE=0.773016 | RMSE=1.034110


In [12]:
# ============================================================
# 13. TabPFN
# ============================================================

tabpfn_model = TabPFNRegressor(
    device="cpu",
    model_path=TABPFN_MODEL_PATH,
    random_state=RANDOM_STATE
)

tabpfn_grid = {}

best_tabpfn, tabpfn_result = train_and_evaluate(
    model_name="TabPFN",
    model=tabpfn_model,
    param_grid=tabpfn_grid,
    n_jobs=1
)



===== TabPFN =====
Best parameters: {}
Five-fold CV MAE: 0.992768
Train: R2=0.969758 | MAE=0.247112 | RMSE=0.369557
Test: R2=0.718934 | MAE=1.080650 | RMSE=1.318600


In [13]:
# ============================================================
# 13. Model comparison
# ============================================================

comparison_results = pd.DataFrame([
    xgb_result,
    ridge_result,
    krr_result,
    rf_result,
    gbdt_result,
    tabpfn_result,
    gpr_result,
])

comparison_results = (
    comparison_results
    .sort_values(
        "Test_R2",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nFinal model comparison:")

print(
    comparison_results[
        [
            "Model",
            "CV_MAE",
            "Test_R2",
            "Test_MAE",
            "Test_RMSE"
        ]
    ]
)

comparison_results.to_csv(
    f"{OUTPUT_DIR}/model_comparison_results.csv",
    index=False
)


Final model comparison:
               Model    CV_MAE   Test_R2  Test_MAE  Test_RMSE
0            XGBoost  0.742416  0.844390  0.757389   0.981133
1  Gradient Boosting  0.910095  0.827132  0.773016   1.034110
2      Random Forest  0.893546  0.782593  1.008776   1.159701
3                GPR  1.085827  0.758492  1.015694   1.222292
4             TabPFN  0.992768  0.718934  1.080650   1.318600
5       Kernel Ridge  1.040894  0.612359  1.355041   1.548545
6              Ridge  1.215305  0.309663  1.872506   2.066520


## c2

In [1]:
# ============================================================
# 1. Import packages
# ============================================================

import os
import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from tabpfn import TabPFNRegressor

from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor
)
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.svm import SVR

from sklearn.gaussian_process import (
    GaussianProcessRegressor
)
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    RBF,
    WhiteKernel
)


D:\app\miniconda\envs\mp311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================
# 2. Basic settings
# ============================================================

DATA_FILE = "data/compound_6_Structural_c2.csv"

TARGET_COL = "c_2"
GROUP_COL = "compound"

N_SELECTED_FEATURES = 14
TEST_SIZE = 0.20
RANDOM_STATE = 6

TABPFN_MODEL_PATH = (
    "tabpfn_models/tabpfn-v2-regressor.ckpt"
)

OUTPUT_DIR = "model_comparison/c2"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

In [3]:
# ============================================================
# 3. Data preparation
# ============================================================

data = pd.read_csv(DATA_FILE)

data = data.drop(
    columns=[
        col for col in data.columns
        if str(col).startswith("Unnamed:")
    ],
    errors="ignore"
)

feature_cols = [
    col for col in data.columns
    if col not in [GROUP_COL, TARGET_COL]
]

aggregation_rules = {
    col: "first"
    for col in feature_cols
}

aggregation_rules[TARGET_COL] = "mean"

data = (
    data
    .groupby(GROUP_COL, as_index=False)
    .agg(aggregation_rules)
)

train_data, test_data = train_test_split(
    data,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

train_data = train_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

X_train_all = train_data[feature_cols]
X_test_all = test_data[feature_cols]

y_train = (
    train_data[TARGET_COL]
    .to_numpy()
)

y_test = (
    test_data[TARGET_COL]
    .to_numpy()
)

print("Total samples:", len(data))
print("Training samples:", len(train_data))
print("Test samples:", len(test_data))
print("Candidate features:", len(feature_cols))

Total samples: 55
Training samples: 44
Test samples: 11
Candidate features: 25


In [4]:
# ============================================================
# 4. XGBoost feature selection
# ============================================================

feature_selector = XGBRegressor(
    n_estimators=100,
    max_depth=2,
    learning_rate=0.05,
    importance_type="gain",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=0
)

feature_selector.fit(
    X_train_all,
    y_train
)

feature_importance = pd.Series(
    feature_selector.feature_importances_,
    index=feature_cols
).sort_values(
    ascending=False
)

selected_features = (
    feature_importance
    .head(N_SELECTED_FEATURES)
    .index
    .tolist()
)

X_train = train_data[selected_features]
X_test = test_data[selected_features]

print("\nSelected features:")

for index, feature in enumerate(
    selected_features,
    start=1
):
    print(f"{index}. {feature}")



Selected features:
1. C_AW
2. AB_AN
3. AB_CR
4. AB_FIE
5. AB_Nd
6. AB_EA
7. AB_SIE
8. C_AN
9. AB_Nv
10. AB_D
11. AB_TC
12. AB_EN
13. AB_AR
14. AB_AW


In [5]:
# ============================================================
# 5. Cross-validation
# ============================================================

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


# ============================================================
# 6. Model training and evaluation function
# ============================================================

def train_and_evaluate(
    model_name,
    model,
    param_grid,
    n_jobs=-1
):

    search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring="neg_mean_absolute_error",
        cv=cv,
        n_jobs=n_jobs,
        refit=True
    )

    search.fit(
        X_train,
        y_train
    )

    best_model = search.best_estimator_

    train_prediction = best_model.predict(
        X_train
    )

    test_prediction = best_model.predict(
        X_test
    )

    train_r2 = r2_score(
        y_train,
        train_prediction
    )

    train_mae = mean_absolute_error(
        y_train,
        train_prediction
    )

    train_rmse = np.sqrt(
        mean_squared_error(
            y_train,
            train_prediction
        )
    )

    test_r2 = r2_score(
        y_test,
        test_prediction
    )

    test_mae = mean_absolute_error(
        y_test,
        test_prediction
    )

    test_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            test_prediction
        )
    )

    print(f"\n===== {model_name} =====")
    print("Best parameters:", search.best_params_)
    print(
        f"Five-fold CV MAE: "
        f"{-search.best_score_:.6f}"
    )
    print(
        f"Train: R2={train_r2:.6f} | "
        f"MAE={train_mae:.6f} | "
        f"RMSE={train_rmse:.6f}"
    )
    print(
        f"Test: R2={test_r2:.6f} | "
        f"MAE={test_mae:.6f} | "
        f"RMSE={test_rmse:.6f}"
    )

    result = {
        "Model": model_name,
        "CV_MAE": -search.best_score_,
        "Train_R2": train_r2,
        "Train_MAE": train_mae,
        "Train_RMSE": train_rmse,
        "Test_R2": test_r2,
        "Test_MAE": test_mae,
        "Test_RMSE": test_rmse,
        "Best_parameters": str(
            search.best_params_
        )
    }

    return best_model, result

In [6]:
# ============================================================
# 7. XGBoost
# ============================================================

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=0
)

xgb_grid = {
    "n_estimators": [
        50,
        100,
        200,
        300,
        400,
    ],
    "max_depth": [
        1,
        2,
        3
    ],
    "learning_rate": [
        0.03,
        0.05,
        0.10
    ]
}

best_xgb, xgb_result = train_and_evaluate(
    model_name="XGBoost",
    model=xgb_model,
    param_grid=xgb_grid
)


===== XGBoost =====
Best parameters: {'learning_rate': 0.05, 'max_depth': 2, 'n_estimators': 300}
Five-fold CV MAE: 3.906545
Train: R2=0.999013 | MAE=0.131119 | RMSE=0.184341
Test: R2=0.753572 | MAE=1.366204 | RMSE=2.039565


In [7]:
# ============================================================
# 8. Ridge
# ============================================================

ridge_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        Ridge()
    )
])

ridge_grid = {
    "model__alpha": [
        0.001,
        0.01,
        0.1,
        1,
        10,
        100
    ]
}

best_ridge, ridge_result = train_and_evaluate(
    model_name="Ridge",
    model=ridge_model,
    param_grid=ridge_grid
)



===== Ridge =====
Best parameters: {'model__alpha': 10}
Five-fold CV MAE: 3.248362
Train: R2=0.320598 | MAE=2.966000 | RMSE=4.836852
Test: R2=-0.181152 | MAE=3.515490 | RMSE=4.465248


In [8]:
# ============================================================
# 9. Gaussian process regression
# ============================================================

gpr_kernel = (
    ConstantKernel(
        constant_value=1.0,
        constant_value_bounds=(1e-2, 1e2)
    )
    * RBF(
        length_scale=1.0,
        length_scale_bounds=(1e-2, 1e2)
    )
    + WhiteKernel(
        noise_level=1e-3,
        noise_level_bounds=(1e-6, 1e0)
    )
)

gpr_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        GaussianProcessRegressor(
            kernel=gpr_kernel,
            normalize_y=True,
            n_restarts_optimizer=5,
            random_state=RANDOM_STATE
        )
    )
])

gpr_grid = {
    "model__alpha": [
        1e-8,
        1e-6,
        1e-4,
        1e-2
    ]
}

best_gpr, gpr_result = train_and_evaluate(
    model_name="GPR",
    model=gpr_model,
    param_grid=gpr_grid
)


===== GPR =====
Best parameters: {'model__alpha': 1e-06}
Five-fold CV MAE: 2.979635
Train: R2=1.000000 | MAE=0.000009 | RMSE=0.000015
Test: R2=0.150440 | MAE=2.464598 | RMSE=3.786953


D:\app\miniconda\envs\mp311\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-06. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


In [9]:
# ============================================================
# 10. Kernel ridge
# ============================================================

krr_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        KernelRidge(
            kernel="rbf"
        )
    )
])

krr_grid = {
    "model__alpha": [
        0.01,
        0.1,
        1,
    ],
    "model__gamma": [
        0.001,
        0.01,
    ]
}

best_krr, krr_result = train_and_evaluate(
    model_name="Kernel Ridge",
    model=krr_model,
    param_grid=krr_grid
)



===== Kernel Ridge =====
Best parameters: {'model__alpha': 0.01, 'model__gamma': 0.01}
Five-fold CV MAE: 3.103628
Train: R2=0.482338 | MAE=2.392371 | RMSE=4.222039
Test: R2=0.258962 | MAE=2.501964 | RMSE=3.536820


In [10]:
# ============================================================
# 11. Random forest
# ============================================================

rf_model = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=1
)

rf_grid = {
    "n_estimators": [
        100,
        300,
        500
    ],
    "max_depth": [
        None,
        3,
        5
    ],
    "min_samples_leaf": [
        1,
        2,
        4
    ]
}

best_rf, rf_result = train_and_evaluate(
    model_name="Random Forest",
    model=rf_model,
    param_grid=rf_grid
)



===== Random Forest =====
Best parameters: {'max_depth': None, 'min_samples_leaf': 4, 'n_estimators': 100}
Five-fold CV MAE: 3.750370
Train: R2=0.470848 | MAE=2.352815 | RMSE=4.268638
Test: R2=0.089848 | MAE=2.831169 | RMSE=3.919672


In [11]:
# ============================================================
# 12. Gradient boosting
# ============================================================

gbdt_model = GradientBoostingRegressor(
    random_state=RANDOM_STATE
)

gbdt_grid = {
    "n_estimators": [
        50,
        100,
        200
    ],
    "learning_rate": [
        0.03,
        0.05,
        0.10
    ],
    "max_depth": [
        1,
        2,
        3
    ],
    "min_samples_leaf": [
        1,
        2,
        4
    ]
}

best_gbdt, gbdt_result = train_and_evaluate(
    model_name="Gradient Boosting",
    model=gbdt_model,
    param_grid=gbdt_grid
)



===== Gradient Boosting =====
Best parameters: {'learning_rate': 0.1, 'max_depth': 3, 'min_samples_leaf': 4, 'n_estimators': 200}
Five-fold CV MAE: 2.442770
Train: R2=0.999940 | MAE=0.029235 | RMSE=0.045424
Test: R2=0.374156 | MAE=1.949133 | RMSE=3.250318


In [12]:
# ============================================================
# 13. TabPFN
# ============================================================

tabpfn_model = TabPFNRegressor(
    device="cpu",
    model_path=TABPFN_MODEL_PATH,
    random_state=RANDOM_STATE
)

tabpfn_grid = {}

best_tabpfn, tabpfn_result = train_and_evaluate(
    model_name="TabPFN",
    model=tabpfn_model,
    param_grid=tabpfn_grid,
    n_jobs=1
)



===== TabPFN =====
Best parameters: {}
Five-fold CV MAE: 2.522066
Train: R2=0.987925 | MAE=0.298928 | RMSE=0.644834
Test: R2=0.606326 | MAE=1.694436 | RMSE=2.577872


In [13]:
# ============================================================
# 14. Model comparison
# ============================================================

comparison_results = pd.DataFrame([
    xgb_result,
    ridge_result,
    krr_result,
    rf_result,
    gbdt_result,
    tabpfn_result,
    gpr_result,
])

comparison_results = (
    comparison_results
    .sort_values(
        "Test_R2",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nFinal model comparison:")

print(
    comparison_results[
        [
            "Model",
            "CV_MAE",
            "Test_R2",
            "Test_MAE",
            "Test_RMSE"
        ]
    ]
)

comparison_results.to_csv(
    f"{OUTPUT_DIR}/model_comparison_results.csv",
    index=False
)


Final model comparison:
               Model    CV_MAE   Test_R2  Test_MAE  Test_RMSE
0            XGBoost  3.906545  0.753572  1.366204   2.039565
1             TabPFN  2.522066  0.606326  1.694436   2.577872
2  Gradient Boosting  2.442770  0.374156  1.949133   3.250318
3       Kernel Ridge  3.103628  0.258962  2.501964   3.536820
4                GPR  2.979635  0.150440  2.464598   3.786953
5      Random Forest  3.750370  0.089848  2.831169   3.919672
6              Ridge  3.248362 -0.181152  3.515490   4.465248
